# A+ (Adjusted ANR) SR — Notebook (bám path gốc)

In [ ]:

import sys, os
parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
if parent not in sys.path:
    sys.path.insert(0, parent)


In [ ]:

import os, cv2, numpy as np
from src.anr_patch import (
    APlusConfig, train_aplus, save_model, load_model,
    predict_image_aplus, train_and_save, run_inference_dir
)

TRAIN_LR_DIR = "../data/degraded_lr"   # LR cho train
TRAIN_HR_DIR = "../data/input_hr"      # HR gốc cho train
TEST_LR_DIR  = "../data/target_lr"     # LR cho test/demo
TEST_HR_DIR  = "../data/target_hr"     # HR cho đánh giá (nếu cần)

CKPT_PATH    = "../checkpoints/anr_aplus.pkl"
OUT_DIR      = "../outputs/anr_aplus"

os.makedirs(os.path.dirname(CKPT_PATH), exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
print("Paths set:")
for k, v in dict(TRAIN_LR_DIR=TRAIN_LR_DIR, TRAIN_HR_DIR=TRAIN_HR_DIR, TEST_LR_DIR=TEST_LR_DIR, TEST_HR_DIR=TEST_HR_DIR, CKPT_PATH=CKPT_PATH, OUT_DIR=OUT_DIR).items():
    print(f"  {k}: {v}")


Paths set:
  TRAIN_LR_DIR: ../data/degraded_lr
  TRAIN_HR_DIR: ../data/input_hr
  TEST_LR_DIR: ../data/target_lr
  TEST_HR_DIR: ../data/target_hr
  CKPT_PATH: ../checkpoints/anr_aplus.pkl
  OUT_DIR: ../outputs/anr_aplus


In [ ]:

SCALE = 2  # 2, 3, hoặc 4

TRAIN_LR_DIR = f"../data/degraded_lr/LR_x{SCALE}"
TRAIN_HR_DIR = f"../data/input_hr/HR_x{SCALE}"
TEST_LR_DIR  = f"../data/target_lr/LR_x{SCALE}"   
TEST_HR_DIR  = f"../data/target_hr/HR_x{SCALE}"   

CKPT_PATH    = f"../checkpoints/anr_aplus_x{SCALE}.pkl"
OUT_DIR      = f"../outputs/anr_aplus_x{SCALE}"


In [9]:
import glob, os
LR = sorted(glob.glob(os.path.join(TRAIN_LR_DIR, "*.png")))
HR = sorted(glob.glob(os.path.join(TRAIN_HR_DIR, "*.png")))
print("LR files:", len(LR))
print("HR files:", len(HR))


LR files: 100
HR files: 100


In [ ]:

cfg = APlusConfig(scale=2, patch_size=7, step=3, n_anchors=1024, pca_dim=30, ridge_lambda=1e-2, rng_seed=42)
model = train_aplus(TRAIN_LR_DIR, TRAIN_HR_DIR, cfg)
save_model(model, CKPT_PATH)
print("Saved:", CKPT_PATH)


Saved: ../checkpoints/anr_aplus_x2.pkl


In [ ]:

run_inference_dir(TEST_LR_DIR, CKPT_PATH, OUT_DIR)
print("Done. Outputs in:", OUT_DIR)


Done. Outputs in: ../outputs/anr_aplus_x2


In [ ]:

import os, re, glob, math, cv2, numpy as np
from src.anr_patch import load_model, predict_image_aplus, rgb2yiq

CKPT = CKPT_PATH              
LR_DIR = TEST_LR_DIR          
HR_DIR = TEST_HR_DIR          
SAVE_PRED = True              

# --- utils ---
def list_imgs(root):
    exts = (".png", ".jpg", ".jpeg", ".bmp")
    return sorted([p for p in glob.glob(os.path.join(root, "**", "*"), recursive=True)
                   if os.path.splitext(p)[1].lower() in exts])

def key_name(p):
    s = os.path.splitext(os.path.basename(p))[0].lower()
    s = re.sub(r'_srf_\d+_(lr|hr)$', '', s)
    s = re.sub(r'_(lr|hr)$', '', s)
    s = re.sub(r'_x\d+$', '', s)
    return s

def to_y(rgb01):
    return rgb2yiq(rgb01)[:, :, 0]

def shave(img, b):
    if b <= 0: return img
    h, w = img.shape[:2]
    return img[b:h-b, b:w-b] if (h > 2*b and w > 2*b) else img

def psnr_y(y1, y2):
    y1 = y1.astype(np.float32); y2 = y2.astype(np.float32)
 
    if y1.max() > 1.5 or y2.max() > 1.5:
        y1 = y1 / 255.0; y2 = y2 / 255.0
    mse = np.mean((y1 - y2)**2)
    return float('inf') if mse == 0 else 10.0 * math.log10(1.0 / mse)

def ssim_y(y1, y2):
    y1 = y1.astype(np.float32); y2 = y2.astype(np.float32)
    if y1.max() > 1.5 or y2.max() > 1.5:
        y1 = y1 / 255.0; y2 = y2 / 255.0
    ksize = (11, 11); sigma = 1.5
    mu1 = cv2.GaussianBlur(y1, ksize, sigma)
    mu2 = cv2.GaussianBlur(y2, ksize, sigma)
    mu1_sq, mu2_sq = mu1*mu1, mu2*mu2
    mu12 = mu1 * mu2
    sigma1_sq = cv2.GaussianBlur(y1*y1, ksize, sigma) - mu1_sq
    sigma2_sq = cv2.GaussianBlur(y2*y2, ksize, sigma) - mu2_sq
    sigma12   = cv2.GaussianBlur(y1*y2, ksize, sigma) - mu12
    C1 = (0.01**2); C2 = (0.03**2)
    ssim_map = ((2*mu12 + C1) * (2*sigma12 + C2)) / ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))
    return float(ssim_map.mean())

model = load_model(CKPT)
S = int(model.cfg.scale) 

lr_all = list_imgs(LR_DIR)
hr_all = list_imgs(HR_DIR)
lr_map = {key_name(p): p for p in lr_all}
hr_map = {key_name(p): p for p in hr_all}
keys = sorted(set(lr_map) & set(hr_map))
print(f"Paired images: {len(keys)} (LR-only: {len(lr_map)-len(keys)}, HR-only: {len(hr_map)-len(keys)})")


psnrs, ssim_scores = [], []
for k in keys:
    lr_path, hr_path = lr_map[k], hr_map[k]

    # predict
    pred_rgb = predict_image_aplus(lr_path, model).astype(np.float32) / 255.0
    hr_bgr = cv2.imread(hr_path, cv2.IMREAD_COLOR)
    hr_rgb = cv2.cvtColor(hr_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0

    H = min(pred_rgb.shape[0], hr_rgb.shape[0])
    W = min(pred_rgb.shape[1], hr_rgb.shape[1])
    pred_rgb = pred_rgb[:H, :W]; hr_rgb = hr_rgb[:H, :W]

    y_pred = shave(to_y(pred_rgb), S)
    y_hr   = shave(to_y(hr_rgb), S)

    p = psnr_y(y_pred, y_hr)
    s = ssim_y(y_pred, y_hr)
    psnrs.append(p); ssim_scores.append(s)

    if SAVE_PRED:
        os.makedirs(OUT_DIR, exist_ok=True)
        out_path = os.path.join(OUT_DIR, os.path.splitext(os.path.basename(lr_path))[0] + "_Aplus.png")
        cv2.imwrite(out_path, cv2.cvtColor((pred_rgb*255.0+0.5).astype(np.uint8), cv2.COLOR_RGB2BGR))

print(f"\n=== RESULTS (Y-channel, shave={S}px) ===")
print(f"PSNR:  mean {np.mean(psnrs):.3f} dB | median {np.median(psnrs):.3f} dB | min {np.min(psnrs):.3f} | max {np.max(psnrs):.3f}")
print(f"SSIM:  mean {np.mean(ssim_scores):.4f} | median {np.median(ssim_scores):.4f} | min {np.min(ssim_scores):.4f} | max {np.max(ssim_scores):.4f}")


Paired images: 5 (LR-only: 0, HR-only: 0)

=== RESULTS (Y-channel, shave=2px) ===
PSNR:  mean 34.656 dB | median 34.349 dB | min 29.682 | max 39.038
SSIM:  mean 0.9446 | median 0.9607 | min 0.8676 | max 0.9831
